In [1]:
import tensorflow as tf
print(tf.__version__)


2.14.0


In [ ]:
import os
import VGG_help1
from VGG_help1 import prepare_dataset, test_on_data, plot_train_history, plot_confusion_matrix, analyze_performance, train_and_evaluate_from_arrays
from VGG_help1 import resnet_model, cv_train_and_evaluate_model, train_and_evaluate_model, cv_train_model, imbalanced_cv_train_and_evaluate_model

In [3]:
import importlib
import VGG_help1
importlib.reload(VGG_help1)


<module 'VGG_help1' from 'c:\\Users\\ZINGA\\Documents\\DDPM_X-Ray-main.21.02.2025\\Codes\\Classification_Models.1.0\\VGG_help1.py'>

#### Usando somente os dados sinteticos

In [ ]:

project_root = r'C:\Users\ZINGA\Documents\DDPM_X-Ray-main.21.02.2025\project_root1'

dataset_dir_generated = os.path.join(project_root, 'generated', 'train')


In [13]:
# -------- TRAIN (70%) --------
train_gen = datagen.flow_from_directory(
    dataset_dir_generated,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=seed
)

# -------- TEMP (30%) --------
temp_gen = datagen.flow_from_directory(
    dataset_dir_generated,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=seed
)

# ============================
# TEMP → VAL + TEST
# ============================
X_temp, y_temp = [], []
steps = math.ceil(temp_gen.samples / batch_size)
for i in range(steps):
    x, y = temp_gen[i]
    X_temp.append(x)
    y_temp.append(y)

X_temp = np.concatenate(X_temp)
y_temp = np.concatenate(y_temp)

split_idx = len(X_temp) // 2
X_val, X_test = X_temp[:split_idx], X_temp[split_idx:]
y_val, y_test = y_temp[:split_idx], y_temp[split_idx:]

# ============================
# EXTRAIR TREINAMENTO COMPLETO DO GENERATOR
# ============================
X_train, y_train = [], []
steps_train = math.ceil(train_gen.samples / batch_size)
for i in range(steps_train):
    x, y = train_gen[i]
    X_train.append(x)
    y_train.append(y)

X_train = np.concatenate(X_train)
y_train = np.concatenate(y_train)

# ============================
# CONVERTENDO PARA ONE-HOT (4 classes)
# ============================
from tensorflow.keras.utils import to_categorical
num_classes = 4

if y_train.ndim == 1:
    y_train = to_categorical(y_train, num_classes)
if y_val.ndim == 1:
    y_val = to_categorical(y_val, num_classes)
if y_test.ndim == 1:
    y_test = to_categorical(y_test, num_classes)


Found 560 images belonging to 4 classes.
Found 240 images belonging to 4 classes.


In [14]:
print("Train samples:", train_gen.samples)
print("Val samples:", len(X_val))
print("Test samples:", len(X_test))
print("Classes:", train_gen.class_indices)


Train samples: 560
Val samples: 120
Test samples: 120
Classes: {'Atelectasis': 0, 'Cardiomegaly': 1, 'Effusion': 2, 'Pneumonia': 3}


In [19]:
classes = ['Atelectasis', 'Cardiomegaly', 'Effusion', 'Pneumonia']

# Todos os pares de 2 classes
from itertools import combinations
pairs = list(combinations(classes, 2))
print(pairs)
# Saída: [('Atelectasis', 'Cardiomegaly'), ('Atelectasis', 'Effusion'), ...]


[('Atelectasis', 'Cardiomegaly'), ('Atelectasis', 'Effusion'), ('Atelectasis', 'Pneumonia'), ('Cardiomegaly', 'Effusion'), ('Cardiomegaly', 'Pneumonia'), ('Effusion', 'Pneumonia')]


In [20]:
# Supondo que você já extraiu X_train, y_train, X_val, y_val, X_test, y_test

def filter_classes(X, y, class_indices, class_pair):
    """
    X, y: arrays completos
    class_indices: dicionário {'Atelectasis':0, ...}
    class_pair: tuple de classes ex: ('Atelectasis','Pneumonia')
    """
    idx1 = class_indices[class_pair[0]]
    idx2 = class_indices[class_pair[1]]
    
    # Se y estiver one-hot
    mask = (y[:, idx1] == 1) | (y[:, idx2] == 1)
    X_filtered = X[mask]
    y_filtered = y[mask][:, [idx1, idx2]]  # pega só as colunas das 2 classes
    return X_filtered, y_filtered


In [28]:
X_train_pair, y_train_pair = filter_classes(X_train, y_train, train_gen.class_indices, pair)
X_val_pair, y_val_pair = filter_classes(X_val, y_val, train_gen.class_indices, pair)
X_test_pair, y_test_pair = filter_classes(X_test, y_test, train_gen.class_indices, pair)

print(f"Classes {pair}:")
print(f"  X_train_pair.shape: {X_train_pair.shape}, y_train_pair.shape: {y_train_pair.shape}")
print(f"  X_val_pair.shape: {X_val_pair.shape}, y_val_pair.shape: {y_val_pair.shape}")
print(f"  X_test_pair.shape: {X_test_pair.shape}, y_test_pair.shape: {y_test_pair.shape}")


Classes ('Cardiomegaly', 'Pneumonia'):
  X_train_pair.shape: (280, 128, 128, 3), y_train_pair.shape: (280, 2)
  X_val_pair.shape: (60, 128, 128, 3), y_val_pair.shape: (60, 2)
  X_test_pair.shape: (60, 128, 128, 3), y_test_pair.shape: (60, 2)


In [22]:
from tensorflow.keras.utils import to_categorical
import numpy as np

num_classes_pair = 2
metrics_all_pairs = []

for pair in pairs:
    print(f"\nTreinando DenseNet com classes: {pair}")
    
    # ---------------------------
    # Filtrar apenas as classes do par
    # ---------------------------
    X_train_pair, y_train_pair = filter_classes(X_train, y_train, train_gen.class_indices, pair)
    X_val_pair, y_val_pair = filter_classes(X_val, y_val, train_gen.class_indices, pair)
    X_test_pair, y_test_pair = filter_classes(X_test, y_test, train_gen.class_indices, pair)
    
    # ---------------------------
    # Verificar se há dados
    # ---------------------------
    print(f"Shapes após filtro:")
    print(f"  X_train_pair: {X_train_pair.shape}, y_train_pair: {y_train_pair.shape}")
    print(f"  X_val_pair:   {X_val_pair.shape}, y_val_pair: {y_val_pair.shape}")
    print(f"  X_test_pair:  {X_test_pair.shape}, y_test_pair: {y_test_pair.shape}")
    
    if len(X_train_pair) == 0 or len(X_val_pair) == 0 or len(X_test_pair) == 0:
        print(f"  [WARNING] Par {pair} ignorado, não há dados suficientes.")
        continue
    
    # ---------------------------
    # Garantir one-hot encoding
    # ---------------------------
    if y_train_pair.ndim == 1:
        y_train_pair = to_categorical(y_train_pair, num_classes_pair)
    if y_val_pair.ndim == 1:
        y_val_pair = to_categorical(y_val_pair, num_classes_pair)
    if y_test_pair.ndim == 1:
        y_test_pair = to_categorical(y_test_pair, num_classes_pair)
    
    # ---------------------------
    # Treinar modelo
    # ---------------------------
    model, metrics, cm = VGG_help1.train_and_evaluate_from_arrays(
        X_train_pair, y_train_pair,
        X_val_pair, y_val_pair,
        X_test_pair, y_test_pair,
        model_fn=VGG_help1.build_densenet_model,
        input_shape=input_shape,
        num_classes=num_classes_pair,
        epochs=epochs,
        batch_size=batch_size,
        title=f"DenseNet_{pair[0]}_{pair[1]}"
    )
    
    print(f"Resultados para {pair}: {metrics}")
    metrics_all_pairs.append(metrics)

# ---------------------------
# Estatísticas gerais
# ---------------------------
if metrics_all_pairs:
    acc_list = [m["Test Accuracy"] for m in metrics_all_pairs]
    mean_acc = np.mean(acc_list)
    std_acc = np.std(acc_list)
    print(f"\nAcurácia média entre todos os pares: {mean_acc:.4f}")
    print(f"Desvio padrão da acurácia: {std_acc:.4f}")
else:
    print("Nenhum par de classes foi treinado com sucesso.")



Treinando DenseNet com classes: ('Atelectasis', 'Cardiomegaly')
Shapes após filtro:
  X_train_pair: (280, 128, 128, 3), y_train_pair: (280, 2)
  X_val_pair:   (120, 128, 128, 3), y_val_pair: (120, 2)
  X_test_pair:  (0, 128, 128, 3), y_test_pair: (0, 2)
  [WARNING] Par ('Atelectasis', 'Cardiomegaly') ignorado, não há dados suficientes.

Treinando DenseNet com classes: ('Atelectasis', 'Effusion')
Shapes após filtro:
  X_train_pair: (280, 128, 128, 3), y_train_pair: (280, 2)
  X_val_pair:   (60, 128, 128, 3), y_val_pair: (60, 2)
  X_test_pair:  (60, 128, 128, 3), y_test_pair: (60, 2)
Epoch 1/5
9/9 [==============================] - 65s 4s/step - loss: 0.9903 - accuracy: 0.6571 - val_loss: 7.8677 - val_accuracy: 0.0000e+00 - lr: 0.0010
Epoch 2/5
9/9 [==============================] - 31s 3s/step - loss: 0.4226 - accuracy: 0.8429 - val_loss: 3.9215e-04 - val_accuracy: 1.0000 - lr: 0.0010
Epoch 3/5
9/9 [==============================] - 44s 5s/step - loss: 0.2239 - accuracy: 0.9179 - val_l

c:\Users\ZINGA\Documents\DDPM_X-Ray-main.21.02.2025\DDPM_X-Ray\Lib\site-packages\sklearn\metrics\_classification.py:386: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


Shapes após filtro:
  X_train_pair: (280, 128, 128, 3), y_train_pair: (280, 2)
  X_val_pair:   (60, 128, 128, 3), y_val_pair: (60, 2)
  X_test_pair:  (60, 128, 128, 3), y_test_pair: (60, 2)
Epoch 1/5
9/9 [==============================] - 53s 4s/step - loss: 0.1819 - accuracy: 0.9071 - val_loss: 17.4984 - val_accuracy: 0.0000e+00 - lr: 0.0010
Epoch 2/5
9/9 [==============================] - 31s 3s/step - loss: 0.0386 - accuracy: 0.9857 - val_loss: 42.8494 - val_accuracy: 0.0000e+00 - lr: 0.0010
Epoch 3/5
9/9 [==============================] - 30s 3s/step - loss: 0.0172 - accuracy: 0.9929 - val_loss: 50.3298 - val_accuracy: 0.0000e+00 - lr: 0.0010
Epoch 4/5
9/9 [==============================] - 30s 3s/step - loss: 0.0665 - accuracy: 0.9857 - val_loss: 40.7292 - val_accuracy: 0.0167 - lr: 0.0010
Epoch 5/5
2/2 [==============================] - 3s 428ms/step
Resultados para ('Cardiomegaly', 'Pneumonia'): {'Test Loss': 0.0, 'Test Accuracy': 1.0}

Treinando DenseNet com classes: ('Effusion

c:\Users\ZINGA\Documents\DDPM_X-Ray-main.21.02.2025\DDPM_X-Ray\Lib\site-packages\sklearn\metrics\_classification.py:386: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


Shapes após filtro:
  X_train_pair: (280, 128, 128, 3), y_train_pair: (280, 2)
  X_val_pair:   (0, 128, 128, 3), y_val_pair: (0, 2)
  X_test_pair:  (120, 128, 128, 3), y_test_pair: (120, 2)
  [WARNING] Par ('Effusion', 'Pneumonia') ignorado, não há dados suficientes.

Acurácia média entre todos os pares: 0.6792
Desvio padrão da acurácia: 0.3946


In [15]:
# ============================
# Extrair X_train e y_train do generator
# ============================
X_train, y_train = [], []

steps_train = math.ceil(train_gen.samples / batch_size)
for i in range(steps_train):
    x, y = train_gen[i]
    X_train.append(x)
    y_train.append(y)

X_train = np.concatenate(X_train)
y_train = np.concatenate(y_train)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)


X_train shape: (560, 128, 128, 3)
y_train shape: (560, 4)


#### Desenet212 (Treinamento apontado)

In [16]:
from tensorflow.keras.utils import to_categorical

# ============================
# CONFIGURAÇÃO DE TREINAMENTO
# ============================
n = 5  # número de runs
epochs = 5
batch_size = 32
input_shape = (128, 128, 3)
num_classes = 4  # agora são 4 classes

title_densenet = "DenseNet121_SyntheticOnly"


# Garantir one-hot (se necessário)
if y_train.ndim == 1:
    y_train = to_categorical(y_train, num_classes)
if y_val.ndim == 1:
    y_val = to_categorical(y_val, num_classes)
if y_test.ndim == 1:
    y_test = to_categorical(y_test, num_classes)

metrics_runs_densenet = []

for i in range(n):
    print(f"DenseNet Run {i+1}/{n}")


    trained_model_densenet, test_metrics_densenet, confusion_matrix_densenet = (
    VGG_help1.train_and_evaluate_from_arrays(
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        X_test=X_test,
        y_test=y_test,
        model_fn=VGG_help1.build_densenet_model,
        input_shape=input_shape,
        num_classes=num_classes,
        epochs=epochs,
        batch_size=batch_size,
        title=f"{title_densenet}_Run{i+1}"
    )
)


    trained_model_densenet.save(
        f"Trained_Models/{title_densenet}_run{i+1}.h5"
    )

    test_metrics_densenet["Config"] = "SyntheticOnly"
    test_metrics_densenet["Run"] = i + 1
    metrics_runs_densenet.append(test_metrics_densenet)

    print(f"Run {i+1} Metrics:", test_metrics_densenet)



DenseNet Run 1/5
Epoch 1/5
18/18 [==============================] - 96s 4s/step - loss: 1.0418 - accuracy: 0.6071 - val_loss: 24.8039 - val_accuracy: 0.0000e+00 - lr: 0.0010
Epoch 2/5
18/18 [==============================] - 74s 4s/step - loss: 0.6920 - accuracy: 0.7036 - val_loss: 26.9271 - val_accuracy: 0.0000e+00 - lr: 0.0010
Epoch 3/5
18/18 [==============================] - 72s 4s/step - loss: 0.5182 - accuracy: 0.7857 - val_loss: 27.5090 - val_accuracy: 0.0000e+00 - lr: 0.0010
Epoch 4/5
18/18 [==============================] - 63s 4s/step - loss: 0.3956 - accuracy: 0.8321 - val_loss: 19.6935 - val_accuracy: 0.0000e+00 - lr: 0.0010
Epoch 5/5
4/4 [==============================] - 4s 519ms/step


c:\Users\ZINGA\Documents\DDPM_X-Ray-main.21.02.2025\DDPM_X-Ray\Lib\site-packages\keras\src\engine\training.py:3079: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Run 1 Metrics: {'Test Loss': 13.649056434631348, 'Test Accuracy': 0.5, 'Config': 'SyntheticOnly', 'Run': 1}
DenseNet Run 2/5
Epoch 1/5
18/18 [==============================] - 88s 4s/step - loss: 1.1280 - accuracy: 0.5946 - val_loss: 21.9852 - val_accuracy: 0.0000e+00 - lr: 0.0010
Epoch 2/5
18/18 [==============================] - 62s 3s/step - loss: 0.7888 - accuracy: 0.6589 - val_loss: 57.6648 - val_accuracy: 0.0000e+00 - lr: 0.0010
Epoch 3/5
18/18 [==============================] - 62s 3s/step - loss: 0.4850 - accuracy: 0.7804 - val_loss: 82.3896 - val_accuracy: 0.0000e+00 - lr: 0.0010
Epoch 4/5
18/18 [==============================] - 63s 4s/step - loss: 0.3881 - accuracy: 0.8321 - val_loss: 46.9910 - val_accuracy: 0.0000e+00 - lr: 0.0010
Epoch 5/5
4/4 [==============================] - 3s 461ms/step


c:\Users\ZINGA\Documents\DDPM_X-Ray-main.21.02.2025\DDPM_X-Ray\Lib\site-packages\keras\src\engine\training.py:3079: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Run 2 Metrics: {'Test Loss': 18.518604278564453, 'Test Accuracy': 0.5, 'Config': 'SyntheticOnly', 'Run': 2}
DenseNet Run 3/5
Epoch 1/5
18/18 [==============================] - 89s 4s/step - loss: 1.1684 - accuracy: 0.5482 - val_loss: 16.4641 - val_accuracy: 0.0000e+00 - lr: 0.0010
Epoch 2/5
18/18 [==============================] - 64s 4s/step - loss: 0.8470 - accuracy: 0.6750 - val_loss: 11.0056 - val_accuracy: 0.0000e+00 - lr: 0.0010
Epoch 3/5
18/18 [==============================] - 60s 3s/step - loss: 0.5875 - accuracy: 0.7536 - val_loss: 7.0292 - val_accuracy: 0.0917 - lr: 0.0010
Epoch 4/5
18/18 [==============================] - 62s 3s/step - loss: 0.4076 - accuracy: 0.8339 - val_loss: 6.3888 - val_accuracy: 0.3917 - lr: 0.0010
Epoch 5/5
4/4 [==============================] - 3s 441ms/step


...

Run 3 Metrics: {'Test Loss': 5.397582054138184, 'Test Accuracy': 0.5, 'Config': 'SyntheticOnly', 'Run': 3}
DenseNet Run 4/5
Epoch 1/5
18/18 [==============================] - 89s 4s/step - loss: 1.1115 - accuracy: 0.6036 - val_loss: 5.5839 - val_accuracy: 0.0583 - lr: 0.0010
Epoch 2/5
18/18 [==============================] - 60s 3s/step - loss: 0.7078 - accuracy: 0.7107 - val_loss: 40.1092 - val_accuracy: 0.0000e+00 - lr: 0.0010
Epoch 3/5
18/18 [==============================] - 61s 3s/step - loss: 0.5777 - accuracy: 0.7518 - val_loss: 10.1552 - val_accuracy: 0.3167 - lr: 0.0010
Epoch 4/5
18/18 [==============================] - 64s 4s/step - loss: 0.4688 - accuracy: 0.8089 - val_loss: 45.2037 - val_accuracy: 0.0000e+00 - lr: 0.0010
Epoch 5/5
4/4 [==============================] - 3s 441ms/step


c:\Users\ZINGA\Documents\DDPM_X-Ray-main.21.02.2025\DDPM_X-Ray\Lib\site-packages\keras\src\engine\training.py:3079: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Run 4 Metrics: {'Test Loss': 3.2561733722686768, 'Test Accuracy': 0.4166666567325592, 'Config': 'SyntheticOnly', 'Run': 4}
DenseNet Run 5/5
Epoch 1/5
18/18 [==============================] - 90s 4s/step - loss: 1.2084 - accuracy: 0.5500 - val_loss: 7.9489 - val_accuracy: 0.0000e+00 - lr: 0.0010
Epoch 2/5
18/18 [==============================] - 60s 3s/step - loss: 0.7585 - accuracy: 0.6661 - val_loss: 46.4370 - val_accuracy: 0.0000e+00 - lr: 0.0010
Epoch 3/5
18/18 [==============================] - 62s 3s/step - loss: 0.5166 - accuracy: 0.7839 - val_loss: 124.8326 - val_accuracy: 0.0000e+00 - lr: 0.0010
Epoch 4/5
18/18 [==============================] - 61s 3s/step - loss: 0.4110 - accuracy: 0.8286 - val_loss: 67.9797 - val_accuracy: 0.0000e+00 - lr: 0.0010
Epoch 5/5
4/4 [==============================] - 4s 444ms/step


c:\Users\ZINGA\Documents\DDPM_X-Ray-main.21.02.2025\DDPM_X-Ray\Lib\site-packages\keras\src\engine\training.py:3079: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Run 5 Metrics: {'Test Loss': 15.720311164855957, 'Test Accuracy': 0.5, 'Config': 'SyntheticOnly', 'Run': 5}


In [17]:
print(hasattr(VGG_help1, "train_and_evaluate_from_arrays"))


True


In [18]:
import numpy as np

# Extrair apenas as acurácias dos testes
test_accuracies = [m["Test Accuracy"] for m in metrics_runs_densenet]

# Estatísticas gerais
mean_acc = np.mean(test_accuracies)
std_acc = np.std(test_accuracies)

print("=== Estatísticas Gerais de Test Accuracy ===")
print(f"Acurácia média: {mean_acc:.4f}")
print(f"Desvio padrão: {std_acc:.4f}")
print(f"Melhor acurácia: {np.max(test_accuracies):.4f}")
print(f"Pior acurácia: {np.min(test_accuracies):.4f}")


=== Estatísticas Gerais de Test Accuracy ===
Acurácia média: 0.4833
Desvio padrão: 0.0333
Melhor acurácia: 0.5000
Pior acurácia: 0.4167


In [58]:
test_losses = [m["Test Loss"] for m in metrics_runs_densenet]
print("=== Estatísticas Gerais de Test Loss ===")
print(f"Loss médio: {np.mean(test_losses):.4f}")
print(f"Desvio padrão: {np.std(test_losses):.4f}")
print(f"Menor loss: {np.min(test_losses):.4f}")
print(f"Maior loss: {np.max(test_losses):.4f}")



=== Estatísticas Gerais de Test Loss ===
Loss médio: 1.3161
Desvio padrão: 1.0889
Menor loss: 0.0728
Maior loss: 3.2896
